In [ ]:
import xarray as xr
from pystac_client import Client
import lazycogs
import stac_geoparquet
import geopandas as gpd
import pandas as pd
import obstore
import re
import numpy as np
import rasterio
import xvec
from functools import reduce
import rioxarray

In [ ]:
csde_stac = Client.open("https://stac.dataspace.copernicus.eu/v1/")
lsp_items = csde_stac.search(
    collections="clms_lsp_global_300m_yearly_v2_cog"
).item_collection()

In [ ]:
lsp_items_dict = lsp_items.to_dict()

In [ ]:
lsp_items_dict["features"][0]

In [ ]:
target_assets = [
    "EOSD-S1",
    "EOSD-S2",
    "SOSD-S1",
    "SOSD-S2",
    "AMPL-S1",
    "AMPL-S2",
    "QA-S1",
    "QA-S2"
]

def sosd_eosd_filter(item: dict) -> bool:
    return any(asset in item["id"] for asset in target_assets)

eosd_sosd_items = list(filter(sosd_eosd_filter, lsp_items_dict["features"]))

In [ ]:
len(eosd_sosd_items)

In [ ]:
for item in eosd_sosd_items:
    # Remove property keys that are not always present and cause failure on conversion
    # to geoparquet.
    item["properties"].pop("platform", None)
    item["properties"].pop("constellation", None)
    # Remove incorrect datetime properties
    item["properties"].pop("datetime", None)
    item["properties"].pop("start_datetime", None)
    item["properties"].pop("end_datetime", None)
    # Parse the correct datetime from the item ID
    dt_str = item["id"].split("_")[3]
    item["properties"]["datetime"] = pd.to_datetime(dt_str, utc=True)
    # Add asset-level proj code so lazycogs knows what's going on
    for asset_key in item["assets"]:
        item["assets"][asset_key]["proj:code"] = "EPSG:4326"

In [ ]:
stac_geoparquet.to_geodataframe(eosd_sosd_items, dtype_backend="numpy_nullable").to_parquet("/tmp/lsp_items.parquet")

In [ ]:
print(list(reduce(set.union, (set(i["assets"].keys()) for i in eosd_sosd_items))))

In [ ]:
parquet_reopen = gpd.read_parquet("/tmp/lsp_items.parquet")

In [ ]:
parquet_reopen.columns

Set up S3 credentials

In [ ]:
import configparser

config = configparser.ConfigParser()
config.read("/home/jovyan/.s3cfg")

access_key = config.get("csde", "access_key")
secret_key = config.get("csde", "secret_key")

store = obstore.store.S3Store(
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
    bucket="eodata",
    endpoint="https://eodata.dataspace.copernicus.eu"
)

In [ ]:
all_detections = gpd.read_file("../data_working/detections_labeled.parquet")
# Toy geometries
my_subtree = all_detections[all_detections.subtree == 100]
my_subtree.explore()

In [ ]:
target_bands = [
    "lsp300_" + asset.lower().replace("-", "_")
    for asset in target_assets
]

In [ ]:
def zonal_stats_driver(geom: gpd.GeoSeries, da: xr.DataArray):
    # Clip box ahead of zonal_stats means that we don't have to rasterize
    # geometry over the entire data extent.
    zs = da.rio.clip_box(*geom.total_bounds, allow_one_dimensional_raster=True)\
        .xvec.zonal_stats(geom, "x", "y")\
        .assign_coords(geometry=geom.index)\
        .to_pandas()

    zs_melt = pd.melt(zs, ignore_index=False, value_name=str(da.band.data))
    return zs_melt

In [ ]:
def zonal_stats_on_asset(asset_name: str, band_name: str) -> pd.DataFrame:
    # Open array. These data are small enough that we can
    # pull the whole thing into RAM.
    target_ids = list(filter(
        lambda x: asset_name in x,
        (i["id"] for i in eosd_sosd_items)
    ))

    lsp_da = lazycogs.open(
        "/tmp/lsp_items.parquet",
        bands=[band_name],
        ids=target_ids,
        store=store,
        crs="EPSG:5071",
        resolution=300,
        max_concurrent_reads=4,
        bbox=all_detections.total_bounds
    ).compute()
    lsp_da = lsp_da.where(lsp_da != -9999).squeeze()

    # Run zonal statistics. This we have to run on each subtree because
    # geometry rasterization is memory intensive.
    this_zs = all_detections.groupby("subtree", group_keys=False)["geometry"].apply(zonal_stats_driver, lsp_da)
    return this_zs

In [ ]:
%%time
# zs_dataframes = []
for (asset, band) in zip(target_assets, target_bands):
    if "QA" not in asset: continue
    print(f"Starting {asset}, {band}")
    this_df = zonal_stats_on_asset(asset, band)
    zs_dataframes.append(this_df)

In [ ]:
# Promote time to an index
zs_dataframes = [df.set_index("time", append=True) for df in zs_dataframes]

In [ ]:
all_zs = pd.concat(zs_dataframes, axis=1, join="inner")

In [ ]:
all_zs.to_parquet("../data_working/detections_phenology.parquet")